# RAG Evaluation Framework with Automated Synthetic Dataset Generation
This notebook provides a complete implementation of RAG Evaluation based on **RAGAS** methodology.

### Evaluation Metrics:
1. **Faithfulness (0.0 to 1.0)**: Checks if the generated answer is derived **ONLY** from the retrieved document context.
2. **Answer Relevancy (0.0 to 1.0)**: Measures how directly the generated response answers the question.
3. **Context Precision (0.0 to 1.0)**: Evaluates if the retriever retrieved relevant context chunks.

### Step 1: Setup & Local LLM Initialization

In [23]:
import os
import json
import random
import pandas as pd
from pathlib import Path
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_chroma import Chroma
from backend.llms.models import LM_STUDIO_MODEL

# Load environment variables from .env
load_dotenv(dotenv_path="/home/preritubuntu/RAG projext/.env")

# Initialize ChatOpenAI pointing to your local LM Studio instance
eval_llm = ChatOpenAI(
    base_url="http://127.0.0.1:1234/v1",
    openai_api_key="lm-studio",
    model=LM_STUDIO_MODEL,
    temperature=0.2
)
print(f"Evaluation LLM ({LM_STUDIO_MODEL}) initialized successfully.")

Evaluation LLM (ornith-1.0-9b) initialized successfully.


### Step 2: Load Document Chunks & Persistent Chroma DB

In [24]:
from backend.loaders.data_loader import load_docling_json
from backend.embeddings.embedding_generator import Embedder
from backend.retriever.retriever_reranking import Retriever

project_root = Path("/home/preritubuntu/RAG projext")
data_dir = project_root / "data"
all_docs = []
json_paths = sorted(data_dir.glob("*.json"))

# Load document chunks from JSON files for BM25 keyword search
for j_path in json_paths:
    try:
        all_docs.extend(load_docling_json(j_path))
    except Exception as e:
        print(f"Error loading {j_path.name}: {e}")

print(f"Loaded {len(all_docs)} total document chunks for retrieval.")

# Load embedding model instance (HuggingFaceEmbeddings)
embedding_model = Embedder.model_loader()

# Connect to primary persistent Chroma DB vector store
vector_store = Chroma(
    embedding_function=embedding_model,
    persist_directory=str(project_root / "chroma_db_vectorstore"),
    collection_name="Multi_doc_vectorstore"
)

# Initialize Hybrid Rerank Retriever
rag_retriever = Retriever()
print("Persistent Chroma DB & RAG Retriever Ready.")

Loaded and partitioned A Comparative Analysis of Gradient Boosting Algorithms [arXiv 1811.12808].json into 187 chunks:
 - Text chunks: 162
 - Table chunks: 2
 - Figure chunks: 23
Loaded and partitioned A ConvNet for the 2020s (ConvNeXt) [arXiv 2201.03545].json into 99 chunks:
 - Text chunks: 83
 - Table chunks: 12
 - Figure chunks: 4
Loaded and partitioned A Critical Review of Recurrent Neural Networks for Sequence Learning [arXiv 1506.00019].json into 132 chunks:
 - Text chunks: 118
 - Table chunks: 0
 - Figure chunks: 14
Loaded and partitioned A Simple Framework for Contrastive Learning of Visual Representations (SimCLR) [arXiv 2002.05709].json into 118 chunks:
 - Text chunks: 86
 - Table chunks: 14
 - Figure chunks: 18
Loaded and partitioned A Survey of Decision Tree Classifier Algorithms [arXiv 1904.03641].json into 63 chunks:
 - Text chunks: 63
 - Table chunks: 0
 - Figure chunks: 0
Loaded and partitioned A Survey of Ensemble Learning for Data Stream Classification [arXiv 1703.048

<All keys matched successfully>


Persistent Chroma DB & RAG Retriever Ready.


### Step 3: Automated Synthetic Dataset Generation via Local LLM
We sample document chunks and prompt `eval_llm` to automatically generate `(question, ground_truth)` pairs based on source context.

In [25]:
NUM_SYNTHETIC_SAMPLES = 10  # Set how many test cases you want to generate (e.g. 10, 50, 100)

# Filter out very short chunks (code/headers) to ensure rich text
valid_chunks = [doc for doc in all_docs if len(doc.page_content.strip()) > 150]
sampled_chunks = random.sample(valid_chunks, min(NUM_SYNTHETIC_SAMPLES, len(valid_chunks)))

eval_test_cases = []
print(f"Generating {len(sampled_chunks)} synthetic QA test cases using local LLM...")

json_parser = JsonOutputParser()

for i, chunk in enumerate(sampled_chunks):
    prompt = f"""You are a University Professor creating an advanced exam question based on technical documents.
Read the context below and generate a specific question and its ground_truth answer based ONLY on the context.

Context:
{chunk.page_content}

Output MUST be a JSON object with keys "question" and "ground_truth".
"""
    try:
        resp = eval_llm.invoke([HumanMessage(content=prompt)])
        data = json_parser.parse(resp.content)
        eval_test_cases.append({
            "question": data["question"],
            "ground_truth": data["ground_truth"],
            "source_chunk": chunk.page_content[:200] + "..."
        })
        print(f"  [{i+1}/{len(sampled_chunks)}] Created Q: '{data['question']}'")
    except Exception as e:
        print(f"  [{i+1}/{len(sampled_chunks)}] Skipped chunk due to parsing error: {e}")

print(f"\nSuccessfully generated {len(eval_test_cases)} synthetic test cases!")

Generating 10 synthetic QA test cases using local LLM...
  [1/10] Created Q: 'According to the provided references, in which conference proceedings was the paper titled 'SQuAD: 100,000+ questions for machine comprehension of text' published?'
  [2/10] Created Q: 'Why did the researchers opt for the 'inverse square root' learning rate schedule instead of the triangular learning rate proposed by Howard and Ruder (2018)?'
  [3/10] Created Q: 'According to the provided text, what dataset and evaluation setting are used to test for potential gender and occupation biases in large language models like Chinchilla, and how is an unbiased model defined within this specific coreference resolution task?'
  [4/10] Created Q: 'Based on the legends provided in subplots (a), (c), and (e) of Figure 7, what are the default configurations for each of the three ablated meta-prompt components—Instruction Ordering, Instruction Scores, and # Exemplars—as indicated by the blue circle markers?'
  [5/10] Create

### Step 4: Run RAG Pipeline Across Generated Test Cases

In [26]:
eval_results = []

for i, test in enumerate(eval_test_cases):
    q = test["question"]
    gt = test["ground_truth"]
    print(f"Executing Test {i+1}: {q}")
    
    # 1. Retrieve top context documents
    retrieved_docs = rag_retriever.start_retriever(all_docs, vector_store, q)
    retrieved_contents = [doc.page_content for doc in retrieved_docs]
    
    # 2. Synthesize answer from LLM
    ai_message = rag_retriever.final_output(q, retrieved_docs)
    generated_ans = ai_message.content if hasattr(ai_message, "content") else str(ai_message)
    
    eval_results.append({
        "question": q,
        "ground_truth": gt,
        "generated_answer": generated_ans,
        "retrieved_contexts": retrieved_contents
    })

print("All synthetic test cases processed.")

Executing Test 1: According to the provided references, in which conference proceedings was the paper titled 'SQuAD: 100,000+ questions for machine comprehension of text' published?
*****************************Final Result of model ************************************
content="\n\nAccording to the provided references, the paper titled 'SQuAD: 100,000+ questions for machine comprehension of text' was published in **Proceedings of the 2016 Conference on Empirical Methods in Natural Language Processing** (EMNLP), pages 2383-2392, Austin, Texas, November 2016 [Source 2, Source 4, Source 5, Source 7]." additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 607, 'prompt_tokens': 3092, 'total_tokens': 3699, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 507, 'rejected_prediction_tokens': None}, 'prompt_tokens_details': None}, 'model_name': 'ornith-1.0-9b', 'system_fingerprint': 'ornith-1.0-9b', 

### Step 5: LLM-as-a-Judge Evaluation Function using LangChain JsonOutputParser

In [27]:
def evaluate_rag_item(item):
    question = item["question"]
    answer = item["generated_answer"]
    contexts = "\n\n".join(item["retrieved_contexts"])
    ground_truth = item["ground_truth"]
    
    parser = JsonOutputParser()
    
    judge_prompt = f"""You are an expert evaluator for RAG systems.
Evaluate the RAG output based on the Question, Contexts, Generated Answer, and Ground Truth.

Question: {question}
Contexts: {contexts}
Generated Answer: {answer}
Ground Truth: {ground_truth}

Score each of the following metrics between 0.0 and 1.0 (where 1.0 is perfect):
1. Faithfulness: Is the answer completely grounded in the Contexts without extra facts?
2. Answer Relevancy: Does the answer directly address the Question?
3. Context Precision: Are the retrieved Contexts relevant and useful for answering the Question?

Output MUST be a JSON object with keys "faithfulness", "answer_relevancy", "context_precision", and "reason".
"""
    
    chain = eval_llm | parser
    
    try:
        scores = chain.invoke([HumanMessage(content=judge_prompt)])
    except Exception as e:
        scores = {"faithfulness": 0.5, "answer_relevancy": 0.5, "context_precision": 0.5, "reason": f"Parsing error: {e}"}
        
    return scores

print("Evaluator Judge chain ready.")

Evaluator Judge chain ready.


### Step 6: Compute Metric Scores and Display Summary Table

In [ ]:
scores_list = []

for res in eval_results:
    scores = evaluate_rag_item(res)
    scores_list.append({
        "Question": res["question"],
        "Faithfulness": scores.get("faithfulness", 0.0),
        "Answer Relevancy": scores.get("answer_relevancy", 0.0),
        "Context Precision": scores.get("context_precision", 0.0),
        "Reason": scores.get("reason", "")
    })

df_results = pd.DataFrame(scores_list)
print("\n=================== RAG Evaluation Summary ===================")
print(df_results[["Question", "Faithfulness", "Answer Relevancy", "Context Precision"]])

print("\n=================== Average Metric Scores ===================")
print(df_results[["Faithfulness", "Answer Relevancy", "Context Precision"]].mean())
df_results